# AML alert triage — final notebook (team 348471A6)

Predicts the probability that a bank-monitoring alert is **escalated** (ROC-AUC metric) from the customer's transaction history.
This notebook is self-contained: it goes from the raw competition files to the validated submission `team_348471A6.csv`.

## Setup (run once)
Tested with **Python 3.14**.

```bash
python3 -m venv .venv
.venv/bin/pip install notebook nbconvert ipykernel      # Jupyter itself
.venv/bin/jupyter notebook final.ipynb                  # or: .venv/bin/jupyter nbconvert --to notebook --execute final.ipynb
```

**macOS only:** LightGBM needs the OpenMP runtime, otherwise `import lightgbm` fails with
`Library not loaded: @rpath/libomp.dylib`. Install it first: `brew install libomp`.

The next cell installs the exact library versions used. Put the four competition files
(`train_signals.csv`, `train_transactions.parquet`, `test_signals.csv`, `test_transactions.parquet`) into the folder `DATA_DIR`.

In [1]:
%pip install -q numpy==2.5.3 pandas==3.0.6 pyarrow==25.0.1 scipy==1.18.1 scikit-learn==1.9.1 lightgbm==4.7.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

# Folder with the 4 competition files. Default: ./data next to this notebook (falls back to ../data inside our repo).
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
OUT_DIR = Path("submissions")          # where team_348471A6.csv is written
TEAM_ID = "348471A6"
SEED = 42

In [3]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

np.random.seed(SEED)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| lightgbm", lgb.__version__)

numpy 2.5.3 | pandas 3.0.6 | scikit-learn 1.9.1 | lightgbm 4.7.0


## 1. Load and clean
Each alert (`signal_id`) has many transactions. We attach the alert date to every transaction and compute how many seconds before
00:00:00 of the alert date it happened.

**Cleaning:** transactions *after* 00:00:00 of the alert date are dropped (840 rows in 21 train alerts, 51 rows in 2 test alerts) —
their timing relative to the alert is ambiguous, so they are treated as potential leakage. Transactions exactly at 00:00:00 are kept.

In [4]:
TYPES = ["karta", "bank_otkazmasi", "naqd", "xalqaro"]   # card, bank transfer, cash, international


def load(split):
    s = pd.read_csv(DATA_DIR / f"{split}_signals.csv", parse_dates=["signal_sanasi"])
    t = pd.read_parquet(DATA_DIR / f"{split}_transactions.parquet")
    t = t.merge(s[["signal_id", "signal_sanasi"]], on="signal_id")
    t["sec_before"] = (t.signal_sanasi - t.tranzaksiya_vaqti).dt.total_seconds()
    n_raw = len(t)
    t = t[t.sec_before >= 0].sort_values(["signal_id", "tranzaksiya_vaqti"]).reset_index(drop=True)
    t["is_in"] = t.kirim_chiqim.eq("kirim").astype(np.int8)
    for ty in TYPES:
        t["is_" + ty] = t.tranzaksiya_turi.eq(ty).astype(np.int8)
    print(f"{split}: {len(s):,} alerts, {n_raw:,} transactions, dropped {n_raw - len(t)} after the alert date")
    return s, t


s_tr, t_tr = load("train")
s_te, t_te = load("test")
print("escalation rate (train):", round(s_tr.eskalatsiya.mean(), 4))

train: 14,000 alerts, 6,987,663 transactions, dropped 840 after the alert date


test: 6,000 alerts, 3,027,575 transactions, dropped 51 after the alert date
escalation rate (train): 0.1718


## 2. Features (110 per alert)
Our EDA showed that simple averages barely separate the classes (every single simple feature had AUC 0.50–0.55), while the *shape* of
amounts and amounts *within each transaction type and direction* matter. Four feature groups, each kept only because it improved
cross-validated AUC by more than 0.002 on every one of 5 different fold splits:

| Group | What | # |
|---|---|---|
| base | count, sum/mean/max/min/std of amount, shares of each type and of incoming | 11 |
| A — amount shape | quantiles 5/25/50/75/95, IQR, tail ratio, skew, kurtosis (all / incoming / outgoing); shares below/above global thresholds; repeated and similar same-day amounts; hits of the per-type amount cap | 33 |
| E — type × direction | for the 8 combinations (e.g. *incoming cash*): count, share, mean amount | 24 |
| F — shape within type × direction | std, min, max, 10/50/90th percentiles, skew for 6 combinations (international skipped: < 50% of alerts have ≥ 5 such transactions; per-alert value is NaN if < 5 transactions) | 42 |

Global amount thresholds for group A (10th/90th/99th percentile of all amounts) come from the raw train transactions only; no target is used anywhere in feature building.

In [5]:
QS = [0.05, 0.25, 0.5, 0.75, 0.95]


def amount_stats(g, prefix):
    a = g.miqdor_indeksi.agg(["size", "sum", "mean", "max", "min", "std"])
    a.columns = [f"{prefix}_{c}" for c in ["n", "amt_sum", "amt_mean", "amt_max", "amt_min", "amt_std"]]
    mix = g[["is_in"] + ["is_" + ty for ty in TYPES]].mean()
    mix.columns = [f"{prefix}_share_{c[3:]}" for c in mix.columns]
    return a.join(mix)


def base_features(t):
    return amount_stats(t.groupby("signal_id"), "all")


# ---------- A. Shape of the amount distribution ----------
def _shape(x, prefix):
    g = x.groupby("signal_id").miqdor_indeksi
    q = g.quantile(QS).unstack()
    q.columns = [f"{prefix}_q{int(c * 100):02d}" for c in QS]
    q[f"{prefix}_iqr"] = q[f"{prefix}_q75"] - q[f"{prefix}_q25"]
    q[f"{prefix}_tail_ratio"] = (q[f"{prefix}_q95"] - q[f"{prefix}_q50"]) / (q[f"{prefix}_q50"] - q[f"{prefix}_q05"])
    d = x.miqdor_indeksi - g.transform("mean")
    m2, m3, m4 = [(d ** k).groupby(x.signal_id).mean() for k in (2, 3, 4)]
    q[f"{prefix}_skew"] = m3 / m2 ** 1.5
    q[f"{prefix}_kurt"] = m4 / m2 ** 2 - 3
    return q


def shape_features(t, thr):
    parts = [_shape(t, "shp_all"), _shape(t[t.is_in == 1], "shp_in"), _shape(t[t.is_in == 0], "shp_out")]
    g = t.groupby("signal_id").miqdor_indeksi
    f = pd.concat(parts, axis=1)
    for name, v in thr.items():
        f[f"shp_share_{name}"] = (t.miqdor_indeksi < v if name.startswith("below") else t.miqdor_indeksi > v).groupby(t.signal_id).mean()
    f["shp_n_repeated_values"] = g.size() - g.nunique()
    r1 = t.assign(a=t.miqdor_indeksi.round(1), d=t.tranzaksiya_vaqti.dt.floor("D"))
    f["shp_max_similar_same_day"] = r1.groupby(["signal_id", "d", "tranzaksiya_turi", "kirim_chiqim", "a"]).size().groupby(level=0).max()
    cap = t.groupby("tranzaksiya_turi").miqdor_indeksi.transform("max")
    f["shp_n_at_type_cap"] = (t.miqdor_indeksi >= cap - 1e-9).groupby(t.signal_id).sum()
    return f


# ---------- E. Type × direction combinations ----------
def combo_features(t):
    combo = t.tranzaksiya_turi + "_" + t.kirim_chiqim
    cnt = pd.crosstab(t.signal_id, combo)
    f = cnt.add_prefix("cmb_n_").join(cnt.div(cnt.sum(axis=1), axis=0).add_prefix("cmb_share_"))
    mean = t.pivot_table(index="signal_id", columns=combo, values="miqdor_indeksi", aggfunc="mean")
    return f.join(mean.add_prefix("cmb_amt_mean_"))


# ---------- F. Amount distribution within each type × direction ----------
F_MIN_TX = 5
F_COMBOS = ["bank_otkazmasi_chiqim", "bank_otkazmasi_kirim", "karta_chiqim", "karta_kirim", "naqd_chiqim", "naqd_kirim"]


def combo_shape_features(t):
    combo = t.tranzaksiya_turi + "_" + t.kirim_chiqim
    x = t[combo.isin(F_COMBOS)].assign(combo=combo)
    g = x.groupby(["signal_id", "combo"]).miqdor_indeksi
    st = g.agg(["size", "std", "min", "max"])
    q = g.quantile([0.1, 0.5, 0.9]).unstack()
    q.columns = ["q10", "q50", "q90"]
    d = x.miqdor_indeksi - g.transform("mean")
    m2, m3 = [(d ** k).groupby([x.signal_id, x.combo]).mean() for k in (2, 3)]
    st = st.join(q).assign(skew=m3 / m2 ** 1.5)
    st = st[st["size"] >= F_MIN_TX].drop(columns="size")
    f = st.unstack("combo")
    f.columns = [f"cmbF_{stat}_{c}" for stat, c in f.columns]
    return f


def build(s, t, thr):
    idx = s.signal_id
    parts = [base_features(t), shape_features(t, thr), combo_features(t), combo_shape_features(t)]
    return pd.concat([p.reindex(idx) for p in parts], axis=1)


# Global thresholds from ALL raw train transactions (as in our frozen pipeline, i.e. before dropping the 840 rows).
a = pd.read_parquet(DATA_DIR / "train_transactions.parquet", columns=["miqdor_indeksi"]).miqdor_indeksi
thr = {"below_q10": a.quantile(0.10), "above_q90": a.quantile(0.90), "above_q99": a.quantile(0.99)}
X_tr, X_te = build(s_tr, t_tr, thr), build(s_te, t_te, thr)
y = s_tr.eskalatsiya.values
assert list(X_tr.columns) == list(X_te.columns)
print("train", X_tr.shape, "| test", X_te.shape)

train (14000, 110) | test (6000, 110)


## 3. Model and cross-validation
**LightGBM** (gradient-boosted decision trees). Parameters were chosen by a 32-configuration grid and confirmed on 5 fold splits:
8 leaves per tree, learning rate 0.01, at least 100 alerts per leaf, 30% of features per tree, L2 = 10.
`scale_pos_weight` compensates for class imbalance (17.2% escalated).

**Validation:** stratified 5-fold CV. Train and test alerts cover the same period (2025-01-01 … 2026-12-31), so random stratified
folds are appropriate. For honesty, one number of trees is used for all folds (the argmax of the *mean* fold AUC curve) instead of
early-stopping each fold on its own validation data. We repeat CV on 5 different fold-split seeds to show that the score is stable.
The number of trees for the final model comes from split seed 42.

In [6]:
PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.01, num_leaves=8, min_child_samples=100,
    lambda_l2=10.0, feature_fraction=0.3, bagging_fraction=0.8, bagging_freq=1,
    seed=SEED, deterministic=True, force_col_wise=True, verbose=-1,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
)
SPLIT_SEEDS = [42, 7, 123, 2024, 31337]


def lgb_cv(X, y, split_seed, max_rounds=6000):
    folds = list(StratifiedKFold(5, shuffle=True, random_state=split_seed).split(np.zeros(len(y)), y))
    res = lgb.cv(PARAMS, lgb.Dataset(X, y), num_boost_round=max_rounds, folds=folds, return_cvbooster=True)
    n_iter = int(np.argmax(res["valid auc-mean"])) + 1
    oof = np.zeros(len(y))
    for (_, va), booster in zip(folds, res["cvbooster"].boosters):
        oof[va] = booster.predict(X.iloc[va], num_iteration=n_iter)
    aucs = [roc_auc_score(y[va], oof[va]) for _, va in folds]
    return dict(split_seed=split_seed, cv_auc_mean=np.mean(aucs), cv_auc_std=np.std(aucs),
                oof_auc=roc_auc_score(y, oof), n_trees=n_iter)


cv = pd.DataFrame([lgb_cv(X_tr, y, ss) for ss in SPLIT_SEEDS])
print(cv.round(4).to_string(index=False))
print(f"\nCV ROC-AUC over 5 split seeds: {cv.cv_auc_mean.mean():.4f} (range {cv.cv_auc_mean.min():.4f}–{cv.cv_auc_mean.max():.4f})")

 split_seed  cv_auc_mean  cv_auc_std  oof_auc  n_trees
         42       0.6465      0.0160   0.6460      508
          7       0.6500      0.0154   0.6493      561
        123       0.6485      0.0134   0.6484      652
       2024       0.6492      0.0104   0.6490      524
      31337       0.6460      0.0158   0.6457      504

CV ROC-AUC over 5 split seeds: 0.6480 (range 0.6460–0.6500)


## 4. Train on the full training set and predict the test set

In [7]:
N_TREES = int(cv.loc[cv.split_seed == SEED, "n_trees"].iloc[0])
model = lgb.train(PARAMS, lgb.Dataset(X_tr, y), num_boost_round=N_TREES)
pred = model.predict(X_te)
imp = pd.Series(model.feature_importance("gain"), X_tr.columns)
print("trees:", N_TREES)
print("top-10 features by gain share:")
print((imp / imp.sum()).sort_values(ascending=False).head(10).round(3).to_string())

trees: 508
top-10 features by gain share:
cmbF_min_bank_otkazmasi_kirim         0.057
cmbF_min_karta_chiqim                 0.049
cmbF_q50_bank_otkazmasi_chiqim        0.043
all_amt_min                           0.037
cmbF_q90_bank_otkazmasi_chiqim        0.035
cmb_amt_mean_bank_otkazmasi_chiqim    0.033
cmb_amt_mean_naqd_kirim               0.031
cmbF_max_karta_chiqim                 0.030
cmbF_max_naqd_kirim                   0.030
shp_out_skew                          0.026


## 5. Validate and write the submission
Checks: columns exactly `signal_id,ehtimollik`; one row per test alert, no duplicates, no missing or unknown IDs; no NaN; all values in [0, 1];
file name `team_348471A6.csv`.

In [8]:
sub = pd.DataFrame({"signal_id": s_te.signal_id, "ehtimollik": pred})
test_ids = pd.read_csv(DATA_DIR / "test_signals.csv").signal_id

assert list(sub.columns) == ["signal_id", "ehtimollik"]
assert len(sub) == len(test_ids) and not sub.signal_id.duplicated().any()
assert set(sub.signal_id) == set(test_ids)
assert sub.ehtimollik.notna().all() and sub.ehtimollik.between(0, 1).all()

OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / f"team_{TEAM_ID}.csv"
sub.to_csv(out, index=False)
check = pd.read_csv(out)
assert list(check.columns) == ["signal_id", "ehtimollik"] and len(check) == len(test_ids)
print(f"OK: wrote {out} — {len(sub)} rows, probabilities in [{sub.ehtimollik.min():.3f}, {sub.ehtimollik.max():.3f}]")

OK: wrote submissions/team_348471A6.csv — 6000 rows, probabilities in [0.175, 0.733]


## References
**Libraries**
- pandas, NumPy, PyArrow — data handling; SciPy — scientific computing (dependency).
- scikit-learn — `StratifiedKFold`, `roc_auc_score`.
- LightGBM — Ke et al., 2017, *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*, NeurIPS. macOS requires `libomp` (Homebrew).
- Also used during development (not needed here): CatBoost (tested, not in the final model), SHAP (model explanations), Matplotlib (charts).

**Ideas and domain knowledge**
- American Express — Default Prediction (Kaggle competition, 2022): the common approach of aggregating each customer's transaction/statement history into one row of statistics. https://www.kaggle.com/competitions/amex-default-prediction
- Feedzai — *Anti-Money Laundering Alert Optimization Using Machine Learning with Graphs*. https://arxiv.org/abs/2112.07508
- Dataiku — AML Alerts Triage solution. https://knowledge.dataiku.com/latest/solutions/financial-services/solution-aml-alerts-triage.html
- FATF (Financial Action Task Force) — money-laundering typology reports (structuring, pass-through, cash and cross-border activity), used as feature ideas. https://www.fatf-gafi.org

**AI tools**
- Claude Code (Anthropic) was used as a coding assistant for EDA, feature engineering, experiments and this notebook. All results were run and checked by the team.